[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/61_continuous_batching_scheduler_solution.ipynb)

# 🔴 Solution: Continuous Batching Scheduler

Reference solution for `continuous_batching_scheduler`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
# ✅ SOLUTION

class ContinuousBatchingScheduler:
    def __init__(self, max_batch_size: int):
        self.max_batch_size = max_batch_size
        self.requests = {}
        self.order = []

    def add_request(self, request_id: str, prompt_tokens, max_new_tokens: int):
        if request_id in self.requests:
            raise ValueError("duplicate request_id")
        self.requests[request_id] = {
            "prompt_tokens": list(prompt_tokens),
            "max_new_tokens": max_new_tokens,
            "generated": 0,
            "prefilled": False,
        }
        self.order.append(request_id)

    def step(self):
        result = {"prefill": [], "decode": [], "finished": []}
        slots = self.max_batch_size

        for request_id in list(self.order):
            if slots == 0:
                break
            req = self.requests[request_id]
            if not req["prefilled"]:
                req["prefilled"] = True
                result["prefill"].append(request_id)
                slots -= 1

        for request_id in list(self.order):
            if slots == 0:
                break
            if request_id not in self.requests:
                continue
            req = self.requests[request_id]
            if req["prefilled"] and req["generated"] < req["max_new_tokens"]:
                req["generated"] += 1
                result["decode"].append(request_id)
                slots -= 1
                if req["generated"] >= req["max_new_tokens"]:
                    result["finished"].append(request_id)
                    self.order.remove(request_id)
                    del self.requests[request_id]
        return result

    def has_pending(self) -> bool:
        return bool(self.requests)


In [ ]:
# Verify
sched = ContinuousBatchingScheduler(max_batch_size=2)
sched.add_request("a", [1, 2], 2)
sched.add_request("b", [3], 1)
while sched.has_pending():
    print(sched.step())


In [ ]:
# Run judge
from torch_judge import check
check('continuous_batching_scheduler')
